# Traffic Signal OpenEnv: LLM-Based Reinforcement Learning Pipeline

This notebook implements a full RL fine-tuning pipeline using Hugging Face **TRL** and **Unsloth**. It orchestrates grid-level traffic policies by interacting with the OpenEnv HTTP API.

In [ ]:
!pip install -q trl unsloth wandb matplotlib requests numpy

In [ ]:
import os
import requests
import numpy as np
import wandb
import torch
from trl import GRPOTrainer, GRPOConfig
from unsloth import FastLanguageModel

# Initialize WandB
wandb.init(project="traffic-openenv-rl")

ENV_URL = "https://guuru-dev-traffic-signal-openenv-2.hf.space"
MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"

In [ ]:
# Load Model with Unsloth optimization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = 2048,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
def reward_fn(prompts, completions, **kwargs):
    """
    Environment-based reward function.
    """
    rewards = []
    for prompt, completion in zip(prompts, completions):
        res = requests.post(f"{ENV_URL}/reset", json={"task_id": "hard_multi", "central_enabled": True})
        state = res.json()
        step_res = requests.post(f"{ENV_URL}/step", json={"action": completion})
        data = step_res.json()
        info = data.get("info", {})
        wandb.log({
            "episode_reward": data.get("reward", 0.0),
            "final_score": info.get("final_score", 0.0),
            "throughput": info.get("throughput", 0.0),
            "mean_queue": info.get("mean_queue", 0.0)
        })
        rewards.append(data.get("reward", 0.0))
    return rewards

training_args = GRPOConfig(
    output_dir="./outputs",
    learning_rate=5e-6,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    max_prompt_length=512,
    max_completion_length=128,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fn,
    args=training_args,
    train_dataset=None, # Load your traffic scenarios here
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
# [Step 5] Save Checkpoint
model.save_pretrained("traffic-llm-checkpoint")
tokenizer.save_pretrained("traffic-llm-checkpoint")

In [ ]:
# [Step 6] Generate Visualizations
import sys
sys.path.append(os.getcwd())
from env.metrics_exporter import generate_training_plots

# Collect history from trainer/wandb and plot
# generate_training_plots(collected_log, "plots")